In [3]:
import re
!pip install pdfplumber
!pip install python-docx
import pdfplumber
from docx import Document
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.1 MB/s eta 0:00:00


In [23]:
# generator = pipeline("any-to-any", model="google/flan-t5-small")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TypeError: Processor was loaded, but it is not an instance of `ProcessorMixin`. Got type `<class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>` instead. Please check that you specified correct pipeline task for the model and model has processor implemented and saved.

In [6]:
def extract_text_from_pdf(file_path):
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                content = page.extract_text()
                if content:
                    text += content
    except Exception as e:
        print("❌ Error reading PDF:", e)
        return None
    return text

In [7]:
def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        text = "\n".join([para.text for para in doc.paragraphs])
        return text
    except Exception as e:
        print("❌ Error reading DOCX:", e)
        return None

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)
    return text

In [9]:
def calculate_ats_score(resume, job_desc):
    cv = CountVectorizer(stop_words='english')
    vectors = cv.fit_transform([resume, job_desc]).toarray()

    resume_vec = vectors[0]
    job_vec = vectors[1]

    match = sum((resume_vec > 0) & (job_vec > 0))
    total = sum(job_vec > 0)

    score = (match / total) * 100 if total != 0 else 0
    return round(score, 2)

In [14]:
def get_missing_keywords(resume, job_desc):
    resume_words = set(resume.split())
    job_words = set(job_desc.split())

    missing = job_words - resume_words
    return list(missing)[:15]

In [15]:
def generate_feedback(resume, job_desc, missing_keywords):

    suggestions = []

    if missing_keywords:
        suggestions.append(f"Add missing skills: {', '.join(missing_keywords[:5])}")

    if "project" not in resume:
        suggestions.append("Include 1-2 strong projects related to the role")

    if "experience" not in resume:
        suggestions.append("Add internships or practical experience")

    if len(resume.split()) < 150:
        suggestions.append("Expand resume with more details and achievements")

    suggestions.append("Add measurable achievements (e.g., improved accuracy by 20%)")
    suggestions.append("Use strong action verbs like Developed, Built, Analyzed")

    return suggestions

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def generate_ai_feedback(resume, job_desc):
    prompt = f"""
    Give 3 short suggestions to improve this resume based on required skills.

    Required Skills: {job_desc}
    Resume: {resume}
    """
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=120,
        num_beams=4,
        early_stopping=True
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

In [17]:
from google.colab import files
uploaded = files.upload()

resume_text = None

if len(uploaded) > 0:
    file_name = list(uploaded.keys())[0]

    if file_name.endswith(".pdf"):
        resume_text = extract_text_from_pdf(file_name)

    elif file_name.endswith(".docx"):
        resume_text = extract_text_from_docx(file_name)

if not resume_text or len(resume_text.strip()) == 0:
    print("⚠️ Using sample resume")
    resume_text = "Python data analysis project experience machine learning"

resume_text = clean_text(resume_text)

Saving HarikaP.docx to HarikaP.docx


In [19]:
def generate_required_skills(role):
    roles = {
        "data analyst": ["python", "sql", "data analysis", "statistics", "excel", "power bi"],
        "data scientist": ["python", "machine learning", "deep learning", "statistics", "pandas", "numpy"],
        "web developer": ["html", "css", "javascript", "react", "frontend", "backend"],
        "ai engineer": ["python", "machine learning", "deep learning", "transformers", "pytorch", "tensorflow"]
    }
    return roles.get(role.lower(), ["python", "problem solving", "communication"])

In [20]:
role = input("Enter Job Role: ")

required_skills = generate_required_skills(role)

# Convert to string for ATS comparison
job_desc = clean_text(" ".join(required_skills))

print("\n🎯 Required Skills:")
for skill in required_skills:
    print("-", skill)

Enter Job Role: data analyst

🎯 Required Skills:
- python
- sql
- data analysis
- statistics
- excel
- power bi


In [21]:
def generate_job_description(role):
    roles = {
        "data analyst": "python sql data analysis statistics visualization excel power bi",
        "data scientist": "python machine learning deep learning statistics pandas numpy",
        "web developer": "html css javascript react frontend backend",
        "ai engineer": "python machine learning deep learning transformers pytorch tensorflow"
    }
    return roles.get(role.lower(), "python problem solving communication")

role = input("Enter Job Role: ")
job_desc = generate_job_description(role)
job_desc = clean_text(job_desc)

print("\nSkills:\n", job_desc)

Enter Job Role: data analyst

Skills:
 python sql data analysis statistics visualization excel power bi


In [22]:
score = calculate_ats_score(resume_text, job_desc)
missing = get_missing_keywords(resume_text, job_desc)
feedback = generate_feedback(resume_text, job_desc, missing)
ai_feedback = generate_ai_feedback(resume_text, job_desc)

print("\n📊 ATS Score:", score, "%")

print("\n❌ Missing Skills:")
for word in missing:
    print("-", word)

print("\n💡 Suggestions:")
for s in feedback:
    print("-", s)

print("\n🤖 AI Suggestions:")
print(ai_feedback)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 ATS Score: 88.89 %

❌ Missing Skills:
- statistics

💡 Suggestions:
- Add missing skills: statistics
- Add measurable achievements (e.g., improved accuracy by 20%)
- Use strong action verbs like Developed, Built, Analyzed

🤖 AI Suggestions:

    Give 3 short suggestions to improve this resume based on required skills.

    Required Skills: python sql data analysis statistics visualization excel power bi
    Resume:                                               posina harika data  analyst                       harikaposina gmail com         9390427670           visakhapatnam  india   https   www linkedin com in harika posina  summary  pre final year artificial intelligence student with a strong interest in data analytics and data driven problem solving  skilled in sql  python  power bi  and excel with hands on project experience  proficient in data cleaning  analysis  and visualization to derive meaningful insights  additionally experienced in html  css  firebase  and ai tools  posses